In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
import json
from tqdm import tqdm
from google.colab import drive


drive.mount('/content/drive')

DRIVE_BASE_DIR = "/content/drive/MyDrive/hangman_project"
if not os.path.exists(DRIVE_BASE_DIR):
    os.makedirs(DRIVE_BASE_DIR)

BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 5e-5
MAX_LENGTH = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SAVE_PATH = os.path.join(DRIVE_BASE_DIR, "hangman_transformer_model")

print(f"Using device: {DEVICE}")


Mounted at /content/drive
Using device: cuda


In [ ]:
TOKENIZER_PATH = os.path.join(DRIVE_BASE_DIR, "tokenizer.json")
with open(TOKENIZER_PATH, "r") as f:
    tokenizer_dict = json.load(f)

id_to_token = {v: k for k, v in tokenizer_dict.items()}
vocab_size = len(tokenizer_dict)

DATASET_PATH = os.path.join(DRIVE_BASE_DIR, "big_dataset.csv")
df = pd.read_csv(DATASET_PATH)
print(f"Loaded dataset with {len(df)} examples")

def encode(text, max_length=MAX_LENGTH):
    token_ids = [tokenizer_dict.get(char, tokenizer_dict.get("<unk>", 0)) for char in text]

    if len(token_ids) < max_length:
        token_ids = token_ids + [tokenizer_dict.get("<pad>", 0)] * (max_length - len(token_ids))
    elif len(token_ids) > max_length:
        token_ids = token_ids[:max_length]

    return token_ids

def decode(token_ids):
    return ''.join([id_to_token.get(id, "<unk>") for id in token_ids])

Loaded dataset with 20208484 examples


In [ ]:
df.dropna(inplace=True)

In [ ]:
df.iloc[15000]

,15728
masked_word,a_duction
current_guess,f
original_word,abduction
wrong_guesses,"[0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, ..."
wrong_count,8
masked_letter,b


In [ ]:

class HangmanDataset(Dataset):
    def __init__(self, dataframe, max_length=MAX_LENGTH):
        self.data = dataframe
        self.max_length = max_length

        self.alphabet = list('abcdefghijklmnopqrstuvwxyz')
        self.char_to_idx = {char: idx for idx, char in enumerate(self.alphabet)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        masked_word = row['masked_word']

        original_word = row['original_word']

        masked_letters = row['masked_letter']
        if isinstance(masked_letters, str):
            masked_letters = list(masked_letters)
        elif not isinstance(masked_letters, list):
            masked_letters = [masked_letters]
        target = torch.zeros(26)
        for letter in masked_letters:
            if isinstance(letter, str) and letter.lower() in self.char_to_idx:
                target[self.char_to_idx[letter.lower()]] = 1
        wrong_guesses = row['wrong_guesses']
        if isinstance(wrong_guesses, str):
            if ' ' in wrong_guesses:
                wrong_guesses = wrong_guesses.split()
            else:
                wrong_guesses = list(wrong_guesses)
        elif not isinstance(wrong_guesses, list):
            wrong_guesses = [wrong_guesses] if not pd.isna(wrong_guesses) else []

        wrong_guesses_feature = torch.zeros(26)
        for letter in wrong_guesses:
            if isinstance(letter, str) and letter.lower() in self.char_to_idx:
                wrong_guesses_feature[self.char_to_idx[letter.lower()]] = 1

        input_ids = torch.tensor(encode(masked_word, self.max_length))
        attention_mask = torch.ones(self.max_length)
        pad_token_id = tokenizer_dict.get("<pad>", 0)
        attention_mask[input_ids == pad_token_id] = 0

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'wrong_guesses': wrong_guesses_feature,
            'target': target,
            'masked_word': masked_word,
            'original_word': original_word
        }

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LENGTH):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class HangmanTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super(HangmanTransformer, self).__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.positional_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.wrong_guesses_proj = nn.Linear(26, d_model)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 26)
        )


    def forward(self, input_ids, attention_mask, wrong_guesses):

        embedded = self.token_embedding(input_ids)

        embedded = self.positional_encoding(embedded)

        encoded = self.transformer_encoder(embedded, src_key_padding_mask=~attention_mask.bool())

        encoded_transposed = encoded.transpose(1, 2)
        pooled = self.pooling(encoded_transposed).squeeze(-1)

        wrong_guesses_embedding = self.wrong_guesses_proj(wrong_guesses)

        combined = torch.cat([pooled, wrong_guesses_embedding], dim=1)
        logits = self.classifier(combined)

        return logits

In [ ]:

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Train set: {len(train_df)} examples")
print(f"Validation set: {len(val_df)} examples")

train_dataset = HangmanDataset(train_df)
val_dataset = HangmanDataset(val_df)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

sample_batch = next(iter(train_dataloader))
print("Sample batch shapes:")
print(f"input_ids: {sample_batch['input_ids'].shape}")
print(f"attention_mask: {sample_batch['attention_mask'].shape}")
print(f"wrong_guesses: {sample_batch['wrong_guesses'].shape}")
print(f"target: {sample_batch['target'].shape}")

model = HangmanTransformer(vocab_size=vocab_size)
model.to(DEVICE)


optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

Train set: 17369307 examples
Validation set: 1929924 examples
Sample batch shapes:
input_ids: torch.Size([32, 20])
attention_mask: torch.Size([32, 20])
wrong_guesses: torch.Size([32, 26])
target: torch.Size([32, 26])


In [ ]:

def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    training_stats = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_accuracy = 0
        train_f1 = 0
        train_samples = 0

        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            wrong_guesses = batch['wrong_guesses'].to(DEVICE)
            targets = batch['target'].to(DEVICE)

            optimizer.zero_grad()

            outputs = model(input_ids, attention_mask, wrong_guesses)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * input_ids.size(0)

            preds = (torch.sigmoid(outputs) > 0.5).float()

            accuracy = (preds == targets).all(dim=1).float().mean().item()
            train_accuracy += accuracy * input_ids.size(0)

            true_positives = (preds * targets).sum(dim=1)
            precision = true_positives / (preds.sum(dim=1) + 1e-10)
            recall = true_positives / (targets.sum(dim=1) + 1e-10)
            f1 = 2 * precision * recall / (precision + recall + 1e-10)
            train_f1 += f1.mean().item() * input_ids.size(0)

            train_samples += input_ids.size(0)


            progress_bar.set_postfix({
                'loss': train_loss / train_samples,
                'acc': train_accuracy / train_samples,
                'f1': train_f1 / train_samples
            })

        model.eval()
        val_loss = 0
        val_accuracy = 0
        val_f1 = 0
        val_samples = 0

        with torch.no_grad():
            progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
            for batch in progress_bar:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                wrong_guesses = batch['wrong_guesses'].to(DEVICE)
                targets = batch['target'].to(DEVICE)

                outputs = model(input_ids, attention_mask, wrong_guesses)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * input_ids.size(0)

                preds = (torch.sigmoid(outputs) > 0.5).float()

                accuracy = (preds == targets).all(dim=1).float().mean().item()
                val_accuracy += accuracy * input_ids.size(0)

                true_positives = (preds * targets).sum(dim=1)
                precision = true_positives / (preds.sum(dim=1) + 1e-10)
                recall = true_positives / (targets.sum(dim=1) + 1e-10)
                f1 = 2 * precision * recall / (precision + recall + 1e-10)
                val_f1 += f1.mean().item() * input_ids.size(0)

                val_samples += input_ids.size(0)

                progress_bar.set_postfix({
                    'loss': val_loss / val_samples,
                    'acc': val_accuracy / val_samples,
                    'f1': val_f1 / val_samples
                })

        train_loss = train_loss / train_samples
        train_accuracy = train_accuracy / train_samples
        train_f1 = train_f1 / train_samples
        val_loss = val_loss / val_samples
        val_accuracy = val_accuracy / val_samples
        val_f1 = val_f1 / val_samples

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, Train F1: {train_f1:.4f} - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}, Val F1: {val_f1:.4f}")

        training_stats.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_accuracy': train_accuracy,
            'train_f1': train_f1,
            'val_loss': val_loss,
            'val_accuracy': val_accuracy,
            'val_f1': val_f1
        })

        checkpoint_path = os.path.join(MODEL_SAVE_PATH, f"checkpoint_epoch_{epoch+1}.pt")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'training_stats': training_stats,
            'vocab_size': vocab_size,
        }, checkpoint_path)
        print(f"Saved checkpoint to {checkpoint_path}")

    return training_stats

if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)

In [ ]:

training_stats = train_model(model, train_dataloader, val_dataloader, optimizer, criterion, EPOCHS)

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'training_stats': training_stats,
    'vocab_size': vocab_size,
}, os.path.join(MODEL_SAVE_PATH, "final_model.pt"))


with open(os.path.join(MODEL_SAVE_PATH, "tokenizer_dict.json"), 'w') as f:
    json.dump(tokenizer_dict, f)

In [11]:
import torch

checkpoint = torch.load('../models/final_model.pt', map_location='cpu')


In [12]:
checkpoint['training_stats']

[{'epoch': 1,
  'train_loss': 0.28242085401008077,
  'train_accuracy': 0.06641445165313863,
  'train_f1': 0.3594803929672898,
  'val_loss': 0.26833123590865693,
  'val_accuracy': 0.10175582043645243,
  'val_f1': 0.4250156762211142},
 {'epoch': 2,
  'train_loss': 0.26940734169237096,
  'train_accuracy': 0.09955480664830338,
  'train_f1': 0.41452746449845584,
  'val_loss': 0.2618383982882522,
  'val_accuracy': 0.12334630793751464,
  'val_f1': 0.44878614476587886},
 {'epoch': 3,
  'train_loss': 0.26519899938044306,
  'train_accuracy': 0.11236648646949558,
  'train_f1': 0.4312419722299877,
  'val_loss': 0.258461659647917,
  'val_accuracy': 0.13390216402303926,
  'val_f1': 0.46031457653669394},
 {'epoch': 4,
  'train_loss': 0.2627146279158484,
  'train_accuracy': 0.12033502545612433,
  'train_f1': 0.4409530088166266,
  'val_loss': 0.2563012864125604,
  'val_accuracy': 0.14284396691268672,
  'val_f1': 0.46940832073663735},
 {'epoch': 5,
  'train_loss': 0.261051305841277,
  'train_accuracy': 